# Contrôle des indicateurs par IRIS (étapes 4 à 6)

Entrées : `data/processed/indicateurs_iris.parquet` (script 07) et `data/exports/carte.gpkg` (script 08).

In [1]:
import geopandas as gpd, pandas as pd
pd.set_option('display.max_columns', 50)
i = gpd.read_parquet('../data/processed/indicateurs_iris.parquet')
pc = i[i.code_departement != '75']
print(i.shape, i.crs.to_string(), '| période :', i.periode_24m.iloc[0])

(5265, 43) EPSG:2154 | période : 2024-01-01 / 2025-12-31


## 1. Couverture des indicateurs (petite couronne)

Une médiane de prix n'est calculée qu'à partir de 5 ventes.

In [2]:
cols = ['revenu_median', 'prix_m2_median_ancien_24m', 'prix_m2_median_vefa_24m', 'evol_prix_m2_ancien',
        'dist_station_actuelle_m', 'part_surface_qpv']
pc.groupby('type_iris')[cols].apply(lambda d: d.notna().mean().round(2))

,revenu_median,prix_m2_median_ancien_24m,prix_m2_median_vefa_24m,evol_prix_m2_ancien,dist_station_actuelle_m,part_surface_qpv
type_iris,,,,,,
A,0.12,0.36,0.09,0.25,1.0,1.0
D,0.13,0.23,0.04,0.17,1.0,1.0
H,0.96,0.85,0.09,0.83,1.0,1.0
Z,0.98,0.25,0.02,0.22,1.0,1.0


In [3]:
pc.groupby('code_departement')[['n_ventes_ancien_24m', 'n_ventes_vefa_24m']].describe().round(0).T

code_departement              77     78     91     92     93     94     95
n_ventes_ancien_24m count  761.0  683.0  507.0  616.0  614.0  530.0  562.0
                    mean    14.0   22.0   24.0   46.0   25.0   35.0   19.0
                    std     28.0   24.0   28.0   31.0   27.0   27.0   23.0
                    min      0.0    0.0    0.0    0.0    0.0    0.0    0.0
                    25%      0.0    1.0    3.0   22.0    5.0   12.0    2.0
                    50%      2.0   14.0   14.0   44.0   16.0   29.0   11.0
                    75%     16.0   34.0   34.0   68.0   36.0   50.0   29.0
                    max    340.0  166.0  236.0  171.0  247.0  136.0  232.0
n_ventes_vefa_24m   count  761.0  683.0  507.0  616.0  614.0  530.0  562.0
                    mean     2.0    1.0    1.0    3.0    2.0    3.0    1.0
                    std      8.0    5.0    4.0   12.0   10.0   10.0    8.0
                    min      0.0    0.0    0.0    0.0    0.0    0.0    0.0
                    25%      0.0    0.0    0.0    0.0    0.0    0.0    0.0
                    50%      0.0    0.0    0.0    0.0    0.0    0.0    0.0
                    75%      0.0    0.0    0.0    0.0    0.0    0.0    0.0
                    max    108.0   63.0   38.0  155.0  116.0  147.0  159.0

## 2. Distributions

In [4]:
pc.groupby('code_departement')[['prix_m2_median_ancien_24m', 'prix_m2_median_vefa_24m', 'evol_prix_m2_ancien',
    'revenu_median', 'dist_station_actuelle_m', 'dist_metro_futur_m']].median().round(3)

,prix_m2_median_ancien_24m,prix_m2_median_vefa_24m,evol_prix_m2_ancien,revenu_median,dist_station_actuelle_m,dist_metro_futur_m
code_departement,,,,,,
77,3152.174,4306.491,-0.042,25450.0,2914.0,30192.0
78,3721.494,4746.868,-0.061,29410.0,1434.0,11479.0
91,2995.192,4179.104,-0.061,26650.0,1316.0,10481.0
92,5915.376,7417.643,-0.085,31880.0,471.0,1415.5
93,3455.757,4800.000,-0.069,18770.0,639.0,1730.0
94,4301.540,5338.983,-0.081,24655.0,649.0,1995.0
95,3148.148,4622.010,-0.049,25255.0,1093.0,10164.0


In [5]:
pc['dist_station_actuelle_m'].describe(percentiles=[0.25, 0.5, 0.75, 0.9]).round(0)

count     4273.0
mean      1971.0
std       2782.0
min         17.0
25%        508.0
50%        958.0
75%       2008.0
90%       5082.0
max      24481.0
Name: dist_station_actuelle_m, dtype: float64

## 3. Cohérence : prix/m² médian par IRIS vs revenu médian

In [6]:
pc[['prix_m2_median_ancien_24m', 'revenu_median']].corr(method='spearman').round(2)

,prix_m2_median_ancien_24m,revenu_median
prix_m2_median_ancien_24m,1.00,0.57
revenu_median,0.57,1.00


## 4. Sélection par défaut (script 08)

Paramètres dans la colonne `parametres`.

In [7]:
e = gpd.read_file('../data/exports/carte.gpkg', layer='iris_indicateurs')
e = e[e.code_departement != '75']
print(e.parametres.iloc[0])
print('IRIS sélectionnés :', int(e.selection.sum()), '/', len(e))
(e[e.selection].groupby(['code_departement', 'nom_commune'])
   .agg(iris=('code_iris', 'size'), prix_m2_ancien=('prix_m2_median_ancien_24m', 'median'),
        revenu=('revenu_median', 'median'), dist_station=('dist_station_actuelle_m', 'median'))
   .sort_values('iris', ascending=False).round(0).head(25))

budget ancien 210000 €, VEFA 290000 €, 45 m², station <= 800 m, revenu >= 0 €
IRIS sélectionnés : 906 / 4273


iris  prix_m2_ancien   revenu  \
code_departement nom_commune                                         
94               Vitry-sur-Seine       29          3737.0  19220.0   
93               Saint-Denis           26          3675.0  17120.0   
                 Aubervilliers         24          3768.0  15470.0   
94               Créteil               19          3235.0  20575.0   
91               Évry-Courcouronnes    15          2044.0  19225.0   
95               Argenteuil            15          2561.0  17600.0   
93               Épinay-sur-Seine      14          2810.0  17480.0   
94               Choisy-le-Roi         13          3571.0  20720.0   
95               Cergy                 13          3009.0  18635.0   
93               Noisy-le-Sec          11          3464.0  18650.0   
95               Sarcelles             10          1814.0  16685.0   
93               La Courneuve          10          2816.0  15285.0   
                 Sevran                10          2144.0  17395.0   
94               Villejuif             10          4181.0  20515.0   
93               Bondy                 10          2873.0  18520.0   
                 Bobigny                9          2831.0  16720.0   
                 Rosny-sous-Bois        9          3864.0  19535.0   
                 Livry-Gargan           9          3169.0  22040.0   
                 Drancy                 9          3179.0  18925.0   
                 Villemomble            8          3559.0  27205.0   
92               Antony                 8          3992.0  29255.0   
91               Viry-Châtillon         8          2478.0  21690.0   
92               Gennevilliers          8          4276.0  19240.0   
93               Aulnay-sous-Bois       8          3693.0  23740.0   
                 Gagny                  8          3197.0  25690.0   

                                     dist_station  
code_departement nom_commune                       
94               Vitry-sur-Seine            444.0  
93               Saint-Denis                276.0  
                 Aubervilliers              477.0  
94               Créteil                    500.0  
91               Évry-Courcouronnes         560.0  
95               Argenteuil                 536.0  
93               Épinay-sur-Seine           308.0  
94               Choisy-le-Roi              261.0  
95               Cergy                      526.0  
93               Noisy-le-Sec               452.0  
95               Sarcelles                  310.0  
93               La Courneuve               256.0  
                 Sevran                     457.0  
94               Villejuif                  394.0  
93               Bondy                      410.0  
                 Bobigny                    291.0  
                 Rosny-sous-Bois            390.0  
                 Livry-Gargan               337.0  
                 Drancy                     554.0  
                 Villemomble                503.0  
92               Antony                     450.0  
91               Viry-Châtillon             536.0  
92               Gennevilliers              276.0  
93               Aulnay-sous-Bois           635.0  
                 Gagny                      598.0